# Capítulo 4 · Una predicción, paso a paso

Seguiremos la pregunta **inicio, inicio, a → n** del nombre ana. Primero hacemos una cuenta manual con valores elegidos para entenderla. Después ejecutamos el modelo real con inicialización y entrenamiento propios. No confundas las cifras ilustrativas con los resultados de aprendizaje.

El cuaderno requiere PyTorch en CPU. En Google Colab ya está instalado; en tu equipo, instálalo con `pip install torch`. No se instala nada al ejecutar estas celdas.


**Entrenamiento:** ana, anna, alba y alan; 19 preguntas para ajustar los parámetros.

**Reserva:** lana y bala, como en los capítulos 2 y 3; diez preguntas para comprobar el resultado al terminar. No ajustamos parámetros con ellas ni elegimos otra configuración según su pérdida.


## De tres fichas a una probabilidad

Inicio se representa con [0.1, −0.2] y a con [0.7, 0.3]. Al unir las tres fichas de inicio, inicio, a, obtenemos seis componentes. Estos valores son ilustrativos.

En la primera unidad elegimos peso 2 para la primera componente y cero para las demás; el sesgo vale 0.1. Aplicamos tanh. En la capa final la salida n recibe el doble de esa activación y las otras cuatro reciben cero.

El código siguiente hace esas mismas cuentas. `x[0]` selecciona la primera componente, `math.tanh` aplica la curva y `math.exp` calcula la exponencial. El denominador suma las cuatro exponenciales iguales a uno y la de n. El coste es el logaritmo natural negativo de la probabilidad de n.


In [1]:
import math
x_manual = [0.1, -0.2, 0.1, -0.2, 0.7, 0.3]
u_manual = 2 * x_manual[0] + 0.1
h_manual = math.tanh(u_manual)
z_n_manual = 2 * h_manual
p_n_manual = math.exp(z_n_manual) / (4 + math.exp(z_n_manual))
coste_manual = -math.log(p_n_manual)
print("Suma:", u_manual)
print("Activación:", h_manual)
print("Logit de n:", z_n_manual)
print("Probabilidad de n:", p_n_manual)
print("Coste:", coste_manual)


Suma: 0.30000000000000004
Activación: 0.29131261245159096
Logit de n: 0.5826252249031819
Probabilidad de n: 0.3092412026392393
Coste: 1.1736337155216467


## Lo que cambia al entrenar

El programa siguiente mantiene la organización que acabamos de recorrer, pero **no utiliza los pesos manuales**. Inicializa sus 111 parámetros con semilla 42 y los corrige con las 19 preguntas de los cuatro nombres. La reserva no se añade al entrenamiento.


## Preparar los nombres y las dos listas de símbolos

Este bloque prepara los datos y los nombres que utilizaremos en el resto del programa. Todavía no hace ninguna predicción.

**Las herramientas.** La primera línea describe el archivo. `math` permite calcular el logaritmo de la referencia uniforme; `time` servirá para medir la duración. `torch` es PyTorch, la biblioteca que maneja los tensores y las derivadas. De ella importamos `nn`, que contiene las capas, y `functional`, al que damos el nombre corto `F`. En ese último módulo está la función de pérdida.

**Los datos.** Las dos tuplas separan entrenamiento y reserva. ENTRADAS enumera los cinco símbolos permitidos en un contexto; SALIDAS enumera las cinco respuestas posibles. Estas cadenas son vocabularios, no ejemplos que se entreguen a la red. Su orden asigna los identificadores.

`enumerate(ENTRADAS)` recorre posiciones y símbolos: primero la posición 0 con inicio, después la 1 con a, y así sucesivamente. La expresión entre llaves guarda la correspondencia en un diccionario. Así, consultar `A_ENTRADA['a']` devuelve 1. El segundo diccionario funciona igual con la lista de salidas; consultar la n devuelve 3.

Las dos últimas líneas fijan el tamaño de la red y el entrenamiento. El contexto tiene tres posiciones; cada ficha, dos componentes; la capa intermedia, ocho unidades. Se harán 400 actualizaciones con tasa 0,1 e inicialización de semilla 42. Estas constantes describen el experimento real, no los pesos elegidos a mano para explicar las cuentas.

In [2]:
"""Capítulo 4: MLP de caracteres. PyTorch en CPU, sin descargar datos."""
import math
import time
import torch
from torch import nn
from torch.nn import functional as F

ENTRENAMIENTO = ("ana", "anna", "alba", "alan")
RESERVA = ("lana", "bala")
ENTRADAS = "^abln"
SALIDAS = "abln$"
A_ENTRADA = {c: i for i, c in enumerate(ENTRADAS)}
A_SALIDA = {c: i for i, c in enumerate(SALIDAS)}
CONTEXTO, DIMENSION, OCULTAS = 3, 2, 8
PASOS, TASA, SEMILLA = 400, 0.1, 42


## Transformar un nombre en filas de entrenamiento

La función recibe nombres y devuelve dos colecciones: los contextos numéricos y sus objetivos. Con ana debe producir las cuatro filas que hemos seguido en papel.

`xs` y `ys` empiezan como listas vacías. En la primera guardaremos entradas; en la segunda, respuestas. El bucle exterior toma un nombre cada vez. La condición rechaza una cadena vacía o cualquier carácter que no esté entre las cuatro letras permitidas. `SALIDAS[:-1]` significa ``todas las salidas salvo la última'', excluyendo el final.

La ventana se inicia como $[0,0,0]$. Multiplicar una lista de un solo cero por tres repite ese elemento. El bucle interior recorre las letras del nombre y una marca final. En cada vuelta, `append` guarda una fila en cada lista. `copy()` guarda una copia de la ventana y el diccionario de salidas convierte la respuesta en su número.

**Sigamos ana.** La primera vuelta guarda contexto $[0,0,0]$ y objetivo 0, que significa a. Después actualiza la ventana a $[0,0,1]$. En la segunda vuelta guarda esa ventana y el objetivo 3, que significa n. Es exactamente nuestra pregunta central. Después continúa con la última a y con el final.

`contexto[1:]` retira la casilla más antigua. La suma de listas añade a la derecha la nueva ficha. No hacemos esa actualización cuando la salida es final, porque no habrá otra predicción dentro de ese nombre.

La comprobación `if not xs` evita devolver un conjunto vacío. Las últimas líneas convierten las listas en tensores enteros de CPU. La función devuelve ambos juntos: por eso podemos escribir `x, y = preparar(...)`. Con los cuatro nombres, x tiene 19 filas de tres números e y tiene 19 respuestas.

In [3]:
def preparar(nombres):
    xs, ys = [], []
    for nombre in nombres:
        if not nombre or any(c not in SALIDAS[:-1] for c in nombre):
            raise ValueError("Nombre vacío o letra desconocida")
        contexto = [A_ENTRADA["^"]] * CONTEXTO
        for siguiente in nombre + "$":
            xs.append(contexto.copy())
            ys.append(A_SALIDA[siguiente])
            if siguiente != "$":
                contexto = contexto[1:] + [A_ENTRADA[siguiente]]
    if not xs:
        raise ValueError("Colección vacía")
    return (torch.tensor(xs, dtype=torch.long, device="cpu"),
            torch.tensor(ys, dtype=torch.long, device="cpu"))


## Construir el recorrido que hace una predicción

Esta función crea la red y la devuelve. Cada línea añade una operación al recorrido de un contexto. Los parámetros se inicializan aquí; todavía no se han corregido con los nombres.

`nn.Sequential` significa que las operaciones se ejecutan una detrás de otra. Al llamar después a `modelo(x)`, PyTorch entrega la salida de cada línea a la siguiente.

**Buscar fichas.** La primera línea crea `Embedding(5, 2)`: cinco fichas, dos números por ficha. Con una entrada $[0,0,1]$ buscará dos veces la ficha de inicio y una vez la de a. Esos contenidos se aprenderán durante el entrenamiento.

**Colocar los seis números.** `Flatten(start_dim=1)` conserva la dimensión de los ejemplos y aplana las dimensiones que vienen después. Para una sola pregunta convierte una tabla de tres parejas en una fila de seis componentes. No mezcla las preguntas entre sí.

**Combinar y transformar.** La primera capa `Linear` recibe seis números y produce ocho sumas ponderadas, con sus sesgos. `Tanh` transforma cada una en una activación. La última capa recibe ocho activaciones y produce cinco logits, uno por salida posible. Los sesgos se incluyen por defecto en las capas lineales.

`.to("cpu")` sitúa los parámetros en CPU. El resultado de esta función es el modelo con sus 111 números ajustables. No incluimos softmax al final porque durante el entrenamiento la función de pérdida recibirá los logits directamente. La transformación a probabilidades aparecerá al generar una letra.

In [4]:
def crear_modelo():
    return nn.Sequential(
        nn.Embedding(len(ENTRADAS), DIMENSION),
        nn.Flatten(start_dim=1),
        nn.Linear(CONTEXTO * DIMENSION, OCULTAS),
        nn.Tanh(),
        nn.Linear(OCULTAS, len(SALIDAS)),
    ).to("cpu")


## Repetir la predicción y la corrección

Aquí hay dos funciones distintas. evaluar devuelve un número de pérdida sin cambiar el modelo. entrenar utiliza la pérdida para modificar los parámetros y devuelve su evolución.

En `evaluar`, `modelo(x)` calcula una fila de logits por contexto. La pérdida compara cada fila con su objetivo de y y promedia los costes. `.item()` convierte ese resultado de un solo valor en un número de Python. `no_grad()` indica que en esta evaluación no necesitaremos derivadas.

En `entrenar` creamos el optimizador. `modelo.parameters()` proporciona las fichas, pesos y sesgos que puede modificar. `lr=TASA` fija el tamaño de las correcciones. El historial guarda primero la pérdida del modelo inicial, asociada al paso cero.

**Una vuelta del bucle.** Primero se descartan los gradientes anteriores con `zero_grad`. Después se calculan los logits y la pérdida actual. `backward()` calcula las derivadas. `step()` las usa para actualizar los parámetros. La siguiente vuelta volverá a predecir con los nuevos valores.

`range(1, PASOS + 1)` recorre del paso 1 al 400, incluidos. Todos los pasos utilizan juntas las 19 preguntas de entrenamiento. La condición del resto de dividir entre 50 reconoce los pasos 50, 100 y los demás múltiplos. En ellos se vuelve a medir la pérdida y se guarda en el historial.

La reserva no aparece en esta función. Tampoco detenemos el entrenamiento cuando una muestra generada parece bonita. Terminamos en el número de pasos fijado. En estas capas no hay un comportamiento distinto para entrenamiento y evaluación; `no_grad` solo evita preparar derivadas cuando no hacen falta.

In [5]:
def evaluar(modelo, x, y):
    with torch.no_grad():
        return F.cross_entropy(modelo(x), y).item()


def entrenar(modelo, x, y):
    optimizador = torch.optim.SGD(modelo.parameters(), lr=TASA)
    historial = [(0, evaluar(modelo, x, y))]
    for paso in range(1, PASOS + 1):
        optimizador.zero_grad(set_to_none=True)
        logits = modelo(x)
        perdida = F.cross_entropy(logits, y)
        perdida.backward()
        optimizador.step()
        if paso % 50 == 0:
            historial.append((paso, evaluar(modelo, x, y)))
    return historial


## Calcular la referencia del capítulo 2

Estas funciones reconstruyen y evalúan el bigrama. Lo hacen con tensores, pero las cuentas son las mismas que en el capítulo 2: contar continuaciones y normalizar cada fila.

`torch.ones` crea una tabla de cinco por cinco llena de unos. Cada uno representa la papeleta añadida por el suavizado. Aún no hemos contado ningún nombre.

`zip(x, y)` toma cada entrada junto con su objetivo. De la entrada solo utiliza `fila[-1]`, su última casilla. Para nuestra pregunta $[0,0,1]$ y objetivo 3, aumenta la casilla de fila 1 y columna 3: después de a, se observó n. Las dos posiciones más antiguas no intervienen, porque esto sigue siendo un bigrama.

La suma con `dim=1` obtiene un total por fila. `keepdim=True` conserva cada total como una columna, para dividir las cinco casillas de esa fila por el mismo número. Por ejemplo, la fila de a suma trece después del suavizado. La división produce su reparto de probabilidades.

En la evaluación, `x[:, -1]` selecciona la última casilla de todas las preguntas. El segundo índice, y, selecciona la salida real de cada una. Así obtenemos una probabilidad por transición, no una nueva predicción elegida al azar.

`log()` calcula el logaritmo natural de cada probabilidad. `mean()` hace la media, el signo menos da el coste y `item()` devuelve el número. Es el mismo procedimiento que ya conocíamos. El suavizado evita las probabilidades cero en esta tabla.

In [6]:
def ajustar_bigramas(x, y):
    cuentas = torch.ones(len(ENTRADAS), len(SALIDAS))
    for fila, objetivo in zip(x, y):
        cuentas[fila[-1], objetivo] += 1
    return cuentas / cuentas.sum(dim=1, keepdim=True)


def evaluar_bigramas(tabla, x, y):
    probabilidades = tabla[x[:, -1], y]
    return -probabilidades.log().mean().item()


## Escribir un nombre con el modelo aprendido

La función recibe un modelo entrenado y un generador aleatorio. Devuelve el texto escrito y una marca que distingue terminar normalmente de alcanzar el límite de pasos.

Comenzamos con tres marcas de inicio y una lista de letras vacía. No estamos aprendiendo, así que el bucle trabaja dentro de `no_grad()`. El límite permite hasta veinte decisiones con el valor por defecto.

`[contexto]` añade los corchetes de una lista de preguntas: hay una sola pregunta con tres identificadores. Convertida en tensor tiene forma $(1,3)$. El modelo devuelve cinco logits en una fila. `softmax(dim=1)` los convierte en un reparto y `[0]` extrae esa única fila.

`torch.multinomial` selecciona un índice al azar según esas probabilidades; el argumento 1 pide una sola elección. `generator=rng` indica qué generador aleatorio debe usar y `item()` extrae el índice del resultado.

**Si sale el índice 0**, `SALIDAS[0]` lo traduce a la letra a. La añadimos al texto y buscamos su ficha de entrada, que es 1, para desplazar la ventana. El nuevo contexto será $[0,0,1]$. Esta traducción entre dos listas es necesaria: copiar directamente el índice seleccionado daría un contexto equivocado.

Si aparece final, unimos las letras con `join` y devolvemos `True`. El símbolo final no se añade. Si se agotan los pasos, devolvemos `False`. Puede haber texto vacío si la primera selección aleatoria ya elige final. No volvemos a seleccionar al azar para ocultarlo.

In [7]:
def generar(modelo, rng, max_pasos=20):
    contexto = [A_ENTRADA["^"]] * CONTEXTO
    letras = []
    with torch.no_grad():
        for _ in range(max_pasos):
            x = torch.tensor([contexto], dtype=torch.long)
            probabilidades = modelo(x).softmax(dim=1)[0]
            indice = torch.multinomial(
                probabilidades, 1, generator=rng).item()
            siguiente = SALIDAS[indice]
            if siguiente == "$":
                return "".join(letras), True
            letras.append(siguiente)
            contexto = contexto[1:] + [A_ENTRADA[siguiente]]
    return "".join(letras), False


## Ejecutar y leer el experimento

La función ejecutar organiza las anteriores. En esta primera parte prepara los datos, entrena y reúne las medidas. Continúa en el bloque de la página siguiente.

Limitamos PyTorch a un hilo de CPU porque el trabajo es muy pequeño. Fijamos la semilla antes de crear la red, para repetir la inicialización. Las llamadas a `preparar` construyen las preguntas de entrenamiento y las de la reserva por separado.

`crear_modelo()` devuelve la red inicial. La llamada a `entrenar` recibe únicamente x e y: los datos de la reserva, xv e yv, no entran ahí. La variable historial recibe las parejas de paso y pérdida que se guardaron durante el aprendizaje.

El reloj se lee antes y después de entrenar. La diferencia mide esa llamada completa, incluido el optimizador y las evaluaciones periódicas. No incluye instalar PyTorch, importarlo o generar los nombres finales. Por eso este tiempo puede ser menor que el que percibes al arrancar todo el archivo.

Después se ajusta el bigrama con x e y. El diccionario `resultados` guarda un nombre y un valor para cada medida: uniforme, bigramas en ambos conjuntos y MLP en ambos conjuntos. Las funciones de evaluación solo consultan los modelos.

El recuento de parámetros suma `numel()`, el número de valores de cada tensor ajustable. Debe dar 111. La última entrada del diccionario guarda los segundos medidos. Hasta aquí hemos reunido las medidas; aún falta imprimirlas y generar muestras.

Las primeras impresiones muestran la versión de PyTorch, las formas de x e y y el historial. Es útil comprobar las formas antes de leer la pérdida: deben aparecer 19 preguntas de tres identificadores y 19 objetivos.

`resultados.items()` permite recorrer los nombres de las medidas junto con sus valores. Después creamos el generador de muestras en CPU con semilla 7. Esta semilla controla las selecciones aleatorias y es independiente de la utilizada al crear los pesos. El generador se prepara fuera del bucle para que avance entre muestras.

En cada una de las diez vueltas, `generar` devuelve texto y una marca. `repr(texto)` imprime el texto entre comillas, haciendo visible una cadena vacía. La expresión condicional elige la palabra ``fin'' o ``truncado''. No imprime únicamente las muestras que se parecen a nombres conocidos.

La función devuelve también el modelo, el historial y las medidas para poder consultarlos en un cuaderno. La condición final sobre `__name__` llama a `ejecutar()` si arrancamos el archivo como programa. Si otro programa lo importa, las funciones quedan disponibles sin iniciar automáticamente el experimento.

Ya están representadas todas las líneas del archivo. Los dos últimos bloques mantienen la sangría porque pertenecen a la misma función. En el cuaderno están reunidos en una sola celda, para que Python reciba la definición completa antes de ejecutarla.

In [8]:
def ejecutar():
    torch.set_num_threads(1)
    torch.manual_seed(SEMILLA)
    x, y = preparar(ENTRENAMIENTO)
    xv, yv = preparar(RESERVA)
    modelo = crear_modelo()
    inicio = time.perf_counter()
    historial = entrenar(modelo, x, y)
    segundos = time.perf_counter() - inicio
    bigramas = ajustar_bigramas(x, y)
    resultados = {
        "uniforme": math.log(len(SALIDAS)),
        "bigramas_entrenamiento": evaluar_bigramas(bigramas, x, y),
        "bigramas_reserva": evaluar_bigramas(bigramas, xv, yv),
        "mlp_entrenamiento": evaluar(modelo, x, y),
        "mlp_reserva": evaluar(modelo, xv, yv),
        "parametros": sum(p.numel() for p in modelo.parameters()),
        "segundos_entrenamiento": segundos,
    }
    print("PyTorch:", torch.__version__, "Dispositivo: CPU")
    print("Formas:", tuple(x.shape), tuple(y.shape))
    print("Historial:", historial)
    for nombre, valor in resultados.items():
        print(nombre, valor)
    rng = torch.Generator(device="cpu").manual_seed(7)
    for _ in range(10):
        texto, completo = generar(modelo, rng)
        print(repr(texto), "fin" if completo else "truncado")
    return modelo, historial, resultados

modelo, historial, resultados = ejecutar()


PyTorch: 2.8.0+cpu Dispositivo: CPU
Formas: (19, 3) (19,)
Historial: [(0, 1.5178232192993164), (50, 1.2839688062667847), (100, 1.0052188634872437), (150, 0.707410991191864), (200, 0.544045627117157), (250, 0.4629152715206146), (300, 0.4168529808521271), (350, 0.38598737120628357), (400, 0.3645842671394348)]
uniforme 1.6094379124341003
bigramas_entrenamiento 1.1139613389968872
bigramas_reserva 1.4099500179290771
mlp_entrenamiento 0.3645842671394348
mlp_reserva 2.043356418609619
parametros 111
segundos_entrenamiento 12.666125699994154
'alan' fin
'alan' fin
'anna' fin
'a' fin
'alan' fin
'alna' fin
'anna' fin
'alba' fin
'alba' fin
'alan' fin


## Ver la pregunta que hemos seguido

Preparamos solo ana. La segunda fila, índice 1 porque se empieza a contar desde cero, debe ser [0,0,1] y su objetivo debe ser 3. Convertimos los tensores a listas o números con `tolist` e `item` para leerlos. El bucle enumera las cuatro preguntas de ese nombre; no vuelve a entrenar.


In [9]:
x_ana, y_ana = preparar(("ana",))
for contexto, objetivo in zip(x_ana, y_ana):
    print(contexto.tolist(), "objetivo:", objetivo.item())


[0, 0, 0] objetivo: 0
[0, 0, 1] objetivo: 3
[0, 1, 4] objetivo: 0
[1, 4, 1] objetivo: 4


## Observar las dimensiones del programa

Utilizamos las 19 preguntas de entrenamiento. Cada capa recibe ese mismo grupo transformado. El bucle imprime el nombre de la capa y la forma de su resultado. `no_grad` evita preparar derivadas para esta observación. El primer tamaño sigue siendo 19. Al buscar las fichas, los datos pasan de dos dimensiones (19,3) a tres (19,3,2).


In [10]:
x, y = preparar(ENTRENAMIENTO)
v = x
with torch.no_grad():
    for capa in modelo:
        v = capa(v)
        print(type(capa).__name__, tuple(v.shape))


Embedding (19, 3, 2)
Flatten (19, 6)
Linear (19, 8)
Tanh (19, 8)
Linear (19, 5)


## Una derivada que se puede comprobar a mano

Este ejemplo es independiente de la red de nombres. `requires_grad=True` indica que queremos la derivada respecto a w. Calculamos u=2w−3 y L=u². `backward` calcula la derivada y `w.grad.item()` muestra el resultado, que debe ser −4. Todavía no se ha modificado w.


In [11]:
w = torch.tensor(1.0, requires_grad=True)
u = 2 * w - 3
perdida = u ** 2
perdida.backward()
print("Gradiente:", w.grad.item())


Gradiente: -4.0


## Ejercicios

1. Escribe los cinco contextos y objetivos que produce `anna`. Mantén separadas la entrada y la respuesta esperada.

2. Traduce inicio, inicio, a a identificadores de entrada. Traduce el objetivo n a su identificador de salida. Explica la función de cada número.

3. Distingue el identificador de una letra y su embedding. Indica cuál permanece fijo y cuál cambia al entrenar.

4. Recupera las tres fichas ilustrativas de inicio, inicio, a y aplánalas. Señala qué dos componentes corresponden a la letra más reciente.

5. Reconstruye los 111 parámetros sin mirar el total: cuenta fichas, pesos y sesgos de cada capa.

6. Sigue la primera unidad del ejemplo manual: pesos $[2,0,0,0,0,0]$, sesgo 0,1 y entrada ilustrativa. Calcula su suma y su activación tanh.

7. Sigue la salida n del ejemplo manual: su único peso no nulo vale 2 y recibe la activación anterior. Calcula su logit, su probabilidad cuando los otros cuatro logits son cero y el coste si n es el objetivo.

8. Explica qué recibe `F.cross_entropy`. Distingue calcular la pérdida de seleccionar al azar la siguiente letra.

9. Para $u=2w-3$ y $L=u^2$, calcula la derivada en $w=1$ y una actualización con tasa 0,1. Relaciona el cálculo con `backward()` y `step()`.

10. Describe las formas `(19, 3)`, `(19, 3, 2)`, `(19, 5)` y `(19,)`. Indica cuántas dimensiones tiene cada tensor y qué significa cada tamaño.

11. Lee la tabla de resultados reales: indica dónde mejora la red y dónde empeora respecto al bigrama. Explica por qué no se contradicen ambos resultados.

12. Supón que la selección aleatoria devuelve el identificador de salida 0. Escribe la letra generada y el identificador que añadimos al contexto. Explica también qué ocurre si sale final o se agota el límite.

## Mi conclusión

Explica con tus palabras el recorrido de inicio, inicio, a hasta la pérdida. Después distingue el ejemplo manual del entrenamiento real y compara las dos columnas de resultados.

Mi respuesta:


## Referencias

El recorrido docente se inspira en [makemore](https://github.com/karpathy/makemore). Las capas y la pérdida se implementan con PyTorch 2.8.0. La bibliografía y las instrucciones completas están en el PDF y en los originales LaTeX del capítulo 4.